# Experiment 11: Reinforcement Learning using Q-Learning (Grid World)


### 1. Import Libraries and Set Seed


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)
print("NumPy version:", np.__version__)
print("Libraries imported successfully.")


### 2. Define the 4x4 Grid World Environment


In [ ]:
GRID_SIZE = 4
START = (0, 0)
GOAL = (3, 3)
WALLS = {(1, 1), (2, 1)}  # blocked cells
ACTIONS = ["U", "R", "D", "L"]
ACTION_DELTAS = [(-1, 0), (0, 1), (1, 0), (0, -1)]

valid_cells = [(r, c) for r in range(GRID_SIZE) for c in range(GRID_SIZE) if (r, c) not in WALLS]
state_to_pos = {i: pos for i, pos in enumerate(valid_cells)}
pos_to_state = {pos: i for i, pos in state_to_pos.items()}
START_STATE = pos_to_state[START]
GOAL_STATE = pos_to_state[GOAL]
N_STATES, N_ACTIONS = len(valid_cells), len(ACTIONS)

def reset():
    return START_STATE

def step(state, action):
    r, c = state_to_pos[state]
    dr, dc = ACTION_DELTAS[action]
    candidate = (r + dr, c + dc)
    
    # Boundary check or collision with wall
    if (candidate[0] < 0 or candidate[0] >= GRID_SIZE or
        candidate[1] < 0 or candidate[1] >= GRID_SIZE or
        candidate in WALLS):
        return state, -5.0, False
    
    next_state = pos_to_state[candidate]
    if next_state == GOAL_STATE:
        return next_state, 10.0, True
    return next_state, -1.0, False

def draw_grid():
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(0, GRID_SIZE)
    ax.set_ylim(0, GRID_SIZE)
    ax.set_xticks(np.arange(0, GRID_SIZE + 1, 1))
    ax.set_yticks(np.arange(0, GRID_SIZE + 1, 1))
    ax.grid(True)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            label = "S" if (r, c) == START else "G" if (r, c) == GOAL else "#" if (r, c) in WALLS else ""
            ax.text(c + 0.5, GRID_SIZE - r - 0.5, label, ha="center", va="center", fontsize=12)
    ax.set_title("4 x 4 Grid World (# = blocked cell)")
    plt.show()

draw_grid()
print(f"Number of valid states: {N_STATES}")
print(f"Number of actions: {N_ACTIONS}")


### 3. Train the Q-Learning Agent (Bellman Equation with ε-Greedy Policy)


In [ ]:
alpha = 0.1
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 2500
max_steps = 100

Q = np.zeros((N_STATES, N_ACTIONS), dtype=float)
episode_rewards = []
episode_steps = []
episode_success = []
epsilon_history = []

def choose_action(state, epsilon_value):
    if np.random.random() < epsilon_value:
        return np.random.randint(N_ACTIONS)  # exploration
    values = Q[state]
    best_actions = np.flatnonzero(np.isclose(values, values.max()))
    return int(np.random.choice(best_actions))  # exploitation

for episode in range(episodes):
    state = reset()
    total_reward = 0.0
    success = False
    
    for t in range(max_steps):
        action = choose_action(state, epsilon)
        next_state, reward, done = step(state, action)
        
        # Q-Learning update: terminal states have no future return
        future_value = 0.0 if done else np.max(Q[next_state])
        target = reward + gamma * future_value
        Q[state, action] += alpha * (target - Q[state, action])
        
        total_reward += reward
        state = next_state
        
        if done:
            success = True
            break
            
    episode_rewards.append(total_reward)
    episode_steps.append(t + 1)
    episode_success.append(int(success))
    epsilon_history.append(epsilon)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print("Training complete.")
print(f"Final epsilon: {epsilon:.3f}")
print(f"Last 100-episode training success rate: {np.mean(episode_success[-100:]) * 100:.1f}%")


### 4. Evaluate Learning Performance (Moving Averages)


In [ ]:
def moving_average(values, window=100):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(moving_average(episode_rewards, 100))
axes[0].set_title("Moving Average: Episode Reward")
axes[0].set_xlabel("Episode (windowed)")
axes[0].set_ylabel("Reward")

axes[1].plot(moving_average(episode_success, 100))
axes[1].set_title("Moving Average: Success Rate")
axes[1].set_xlabel("Episode (windowed)")
axes[1].set_ylabel("Success fraction")
axes[1].set_ylim(-0.05, 1.05)

axes[2].plot(epsilon_history)
axes[2].set_title("Exploration Probability ε")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("ε")
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()


### 5. Test the Learned Greedy Policy (Zero Exploration)


In [ ]:
def evaluate_policy(n_eval=200):
    successes = 0
    rewards = []
    lengths = []
    paths = []
    
    for _ in range(n_eval):
        state = reset()
        total_reward = 0.0
        path = [state]
        done = False
        
        for t in range(max_steps):
            best_actions = np.flatnonzero(np.isclose(Q[state], Q[state].max()))
            action = int(np.random.choice(best_actions))
            next_state, reward, done = step(state, action)
            total_reward += reward
            state = next_state
            path.append(state)
            if done:
                break
                
        successes += int(done)
        rewards.append(total_reward)
        lengths.append(len(path) - 1)
        paths.append(path)
        
    return {
        "success_rate": successes / n_eval,
        "mean_reward": float(np.mean(rewards)),
        "mean_steps": float(np.mean(lengths)),
        "sample_path": paths[0]
    }

evaluation = evaluate_policy()
print(f"Evaluation episodes: 200")
print(f"Success rate: {evaluation['success_rate'] * 100:.1f}%")
print(f"Average reward: {evaluation['mean_reward']:.2f}")
print(f"Average steps per episode: {evaluation['mean_steps']:.2f}")
print("One sample state path:", [state_to_pos[s] for s in evaluation["sample_path"]])


### 6. Display the Learned Q-Table


In [ ]:
q_table_df = pd.DataFrame(
    Q,
    columns=[f"Action {a}" for a in ACTIONS],
    index=[f"State {s} {state_to_pos[s]}" for s in range(N_STATES)]
)
display(q_table_df.round(2))


### 7. Extract and Visualize the Learned Greedy Policy


In [ ]:
ARROWS = {"U": "↑", "R": "→", "D": "↓", "L": "←"}

policy = {}
for s in range(N_STATES):
    if s == GOAL_STATE:
        policy[s] = "G"
    else:
        best = np.flatnonzero(np.isclose(Q[s], Q[s].max()))
        policy[s] = ARROWS[ACTIONS[int(best[0])]]

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(0, GRID_SIZE)
ax.set_ylim(0, GRID_SIZE)
ax.set_xticks(np.arange(0, GRID_SIZE + 1, 1))
ax.set_yticks(np.arange(0, GRID_SIZE + 1, 1))
ax.grid(True, linewidth=1.5)
ax.set_xticklabels([])
ax.set_yticklabels([])

for r in range(GRID_SIZE):
    for c in range(GRID_SIZE):
        pos = (r, c)
        if pos == START:
            label = "S\n" + policy[pos_to_state[pos]]
        elif pos == GOAL:
            label = "G"
        elif pos in WALLS:
            label = "#"
        else:
            label = policy[pos_to_state[pos]]
        ax.text(c + 0.5, GRID_SIZE - r - 0.5, label, ha="center", va="center", fontsize=12)

ax.set_title("Greedy Policy Learned by Q-Learning")
plt.show()

print("Policy legend: ↑ Up, → Right, ↓ Down, ← Left, G Goal, # Wall")


### 8. Study Exploration-Exploitation Trade-off Across Training Phases


In [ ]:
def summarize_window(start, end, label):
    return {
        "Training phase": label,
        "Episodes": f"{start + 1}-{end}",
        "Success rate (%)": round(100 * np.mean(episode_success[start:end]), 1),
        "Mean reward": round(float(np.mean(episode_rewards[start:end])), 2),
        "Mean steps": round(float(np.mean(episode_steps[start:end])), 2),
        "Mean epsilon": round(float(np.mean(epsilon_history[start:end])), 3)
    }

summary = pd.DataFrame([
    summarize_window(0, 100, "Early"),
    summarize_window(episodes // 2 - 50, episodes // 2 + 50, "Middle"),
    summarize_window(episodes - 100, episodes, "Late")
])
display(summary)
